In [12]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import math
import torch.nn as nn
import torch.nn.functional as F

In [13]:
SEED = 1234
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4

# Si está en True, proyectamos los espectros generados al subespacio
# de Fourier compatible con imágenes reales. Esto ayuda a preservar
# la simetría hermítica durante el proceso reverse.
ENFORCE_REAL_SPECTRUM = True

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def get_device():
    if torch.cuda.is_available():
        print("Usando GPU:", torch.cuda.get_device_name(0))
        return torch.device("cuda")
    else:
        print("Usando CPU")
        return torch.device("cpu")

device = get_device()

Usando GPU: Quadro RTX 6000


In [14]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) #Normalización de los datos de MNIST
])

In [15]:
dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [16]:
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

In [17]:
x0, labels = next(iter(loader))

print("1. x0 antes de moverlo a la GPU:", x0.shape) #Datos de MNIST están en escala de grises, por lo que solo hay un canal

1. x0 antes de moverlo a la GPU: torch.Size([64, 1, 28, 28])


In [18]:
def image_to_fourier(x):
    """
    x: [B, 1, H, W] real

    return:
    [B, 2, H, W]
    canal 0 = parte real
    canal 1 = parte imaginaria
    """

    # Transformada de Fourier 2D
    X = torch.fft.fft2(
        x,
        norm="ortho"
    )

    # Movemos las frecuencias bajas al centro
    X = torch.fft.fftshift(
        X,
        dim=(-2, -1)
    )

    # Separamos parte real e imaginaria
    X_real = X.real
    X_imag = X.imag

    X_ri = torch.cat(
        [X_real, X_imag],
        dim=1
    )

    return X_ri

In [19]:
def fourier_to_image(X_ri):
    """
    X_ri: [B, 2, H, W]

    return:
    imagen reconstruida [B, 1, H, W]
    """

    real = X_ri[:, 0:1]
    imag = X_ri[:, 1:2]

    # Reconstruir tensor complejo
    X = torch.complex(real, imag)

    # Deshacer fftshift
    X = torch.fft.ifftshift(
        X,
        dim=(-2, -1)
    )

    # Transformada inversa
    x = torch.fft.ifft2(
        X,
        norm="ortho"
    )

    return x.real

x0, _ = next(iter(loader))

x0 = x0[:1].to(device)

X0 = image_to_fourier(x0)

x_reconstructed = fourier_to_image(X0)

print("Imagen:", x0.shape)
print("Fourier:", X0.shape)
print("Reconstruida:", x_reconstructed.shape)

error = torch.mean(
    (x0 - x_reconstructed) ** 2
)

print("Error de reconstrucción:", error.item())

Imagen: torch.Size([1, 1, 28, 28])
Fourier: torch.Size([1, 2, 28, 28])
Reconstruida: torch.Size([1, 1, 28, 28])
Error de reconstrucción: 1.8499077595289515e-14


In [20]:
def fourier_to_complex_image(X_ri):
    """
    Convierte una representación Fourier de dos canales a una imagen
    compleja mediante la transformada inversa.

    X_ri:
        [B, 2, H, W]

    return:
        tensor complejo [B, 1, H, W]
    """

    real = X_ri[:, 0:1]
    imag = X_ri[:, 1:2]

    # Reconstruir tensor complejo.
    X = torch.complex(real, imag)

    # Deshacer fftshift.
    X = torch.fft.ifftshift(
        X,
        dim=(-2, -1)
    )

    # Transformada inversa de Fourier.
    x = torch.fft.ifft2(
        X,
        norm="ortho"
    )

    return x


In [21]:
def project_to_real_fourier(X_ri):
    """
    Proyecta un espectro arbitrario de dos canales al subespacio de
    espectros que corresponden a imágenes reales.

    Esto equivale a:
        Fourier -> imagen compleja -> tomar parte real -> Fourier

    De esta forma se recupera automáticamente la simetría hermítica.
    """

    x_real = fourier_to_image(X_ri)
    return image_to_fourier(x_real)

In [22]:
def fourier_magnitude(X_ri):
    """
    Devuelve log(1 + |X|) para visualizar el espectro de Fourier.
    """

    real = X_ri[:, 0:1]
    imag = X_ri[:, 1:2]

    magnitude = torch.sqrt(real**2 + imag**2)

    return torch.log1p(magnitude)


Prueba fourier

In [23]:
x0, _ = next(iter(loader))

x0 = x0[:1].to(device)

X0 = image_to_fourier(x0)

x_reconstructed_complex = fourier_to_complex_image(X0)
x_reconstructed = x_reconstructed_complex.real

print("Imagen:", x0.shape)
print("Fourier:", X0.shape)
print("Reconstruida:", x_reconstructed.shape)

error = torch.mean(
    (x0 - x_reconstructed) ** 2
)

imaginary_residual = torch.mean(
    torch.abs(x_reconstructed_complex.imag)
)

print("Error de reconstrucción:", error.item())
print(
    "Residuo imaginario después de IFFT:",
    imaginary_residual.item()
)

# Visualización de la transformación ida y vuelta.
magnitude = fourier_magnitude(X0)[0, 0].detach().cpu()

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
image_original = (x0[0, 0].detach().cpu() + 1) / 2
image_original = image_original.clamp(0, 1)
plt.imshow(image_original, cmap="gray")
plt.title("Imagen original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(magnitude, cmap="gray")
plt.title("Magnitud de Fourier")
plt.axis("off")

plt.subplot(1, 3, 3)
image_reconstructed = (
    x_reconstructed[0, 0].detach().cpu() + 1
) / 2
image_reconstructed = image_reconstructed.clamp(0, 1)
plt.imshow(image_reconstructed, cmap="gray")
plt.title("Reconstrucción")
plt.axis("off")

plt.tight_layout()

plt.savefig(
    "fourier_roundtrip.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Prueba Fourier ida-vuelta guardada.")


Imagen: torch.Size([1, 1, 28, 28])
Fourier: torch.Size([1, 2, 28, 28])
Reconstruida: torch.Size([1, 1, 28, 28])
Error de reconstrucción: 1.9798920803618808e-14
Residuo imaginario después de IFFT: 4.9524288669999805e-08
Prueba Fourier ida-vuelta guardada.


Proceso Foward

In [24]:
T = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(
    beta_start,
    beta_end,
    T,
    device=device
)

alphas = 1.0 - betas

alpha_bars = torch.cumprod(
    alphas,
    dim=0
)


def forward_diffusion_fourier(X0, t):
    """
    Proceso forward directamente en el espacio de Fourier.

    X0:
        [B, 2, H, W]

    t:
        [B]

    La idea es partir del mismo proceso espacial:

        x_t = sqrt(alpha_bar_t) x_0
              + sqrt(1-alpha_bar_t) epsilon

    y aplicar Fourier. Como Fourier es lineal:

        X_t = sqrt(alpha_bar_t) X_0
              + sqrt(1-alpha_bar_t) F(epsilon)

    Por eso generamos el ruido primero en espacio real y luego lo
    transformamos a Fourier.
    """

    B, _, H, W = X0.shape

    # Ruido gaussiano en espacio real.
    spatial_noise = torch.randn(
        B,
        1,
        H,
        W,
        device=X0.device
    )

    # El mismo ruido llevado al espacio de Fourier.
    fourier_noise = image_to_fourier(spatial_noise)

    alpha_bar_t = alpha_bars[t]

    alpha_bar_t = alpha_bar_t[:, None, None, None]

    Xt = (  # Proceso de difusión hacia adelante en Fourier
        torch.sqrt(alpha_bar_t) * X0
        +
        torch.sqrt(1 - alpha_bar_t) * fourier_noise
    )

    return Xt, fourier_noise


Prueba de forward en Fourier

In [25]:
x0, _ = next(iter(loader))
x0 = x0.to(device)

X0 = image_to_fourier(x0)

t = torch.randint(
    0,
    T,
    (x0.shape[0],),
    device=device
)

Xt, fourier_noise = forward_diffusion_fourier(
    X0,
    t
)

print("2. x0 después de moverlo a la GPU:", x0.shape)
print("3. X0 en Fourier:", X0.shape)
print("4. Xt después de difusión forward en Fourier:", Xt.shape)
print("5. Ruido Fourier generado:", fourier_noise.shape)

2. x0 después de moverlo a la GPU: torch.Size([64, 1, 28, 28])
3. X0 en Fourier: torch.Size([64, 2, 28, 28])
4. Xt después de difusión forward en Fourier: torch.Size([64, 2, 28, 28])
5. Ruido Fourier generado: torch.Size([64, 2, 28, 28])


Visualización del foward

In [26]:
x0, _ = next(iter(loader))
x0 = x0[:1].to(device)

X0 = image_to_fourier(x0)

times = [0, 100, 300, 600, 999]

# Fila 1: magnitud en Fourier
# Fila 2: la misma muestra reconstruida al espacio real
plt.figure(figsize=(14, 6))

for i, timestep in enumerate(times):

    t = torch.tensor(
        [timestep],
        device=device
    )

    Xt, _ = forward_diffusion_fourier(
        X0,
        t
    )

    magnitude = fourier_magnitude(Xt)[0, 0].detach().cpu()

    reconstructed_xt = fourier_to_image(Xt)

    image = (
        reconstructed_xt[0, 0].detach().cpu() + 1
    ) / 2
    image = image.clamp(0, 1)

    # Magnitud del espectro.
    plt.subplot(2, len(times), i + 1)
    plt.imshow(magnitude, cmap="gray")
    plt.title(f"Fourier t={timestep}")
    plt.axis("off")

    # Reconstrucción espacial.
    plt.subplot(
        2,
        len(times),
        len(times) + i + 1
    )
    plt.imshow(image, cmap="gray")
    plt.title(f"IFFT t={timestep}")
    plt.axis("off")

plt.tight_layout()

plt.savefig(
    "fourier_forward_process.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Proceso forward en Fourier guardado.")

Proceso forward en Fourier guardado.


Time embedding

In [27]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2

        frequencies = math.log(10000) / (half_dim - 1)

        frequencies = torch.exp(
            torch.arange(
                half_dim,
                device=t.device
            ) * -frequencies
        )

        angles = t.float()[:, None] * frequencies[None, :]

        embedding = torch.cat(
            [angles.sin(), angles.cos()],
            dim=1
        )

        return embedding


time_embedding = SinusoidalTimeEmbedding(64).to(device)

test_t = torch.tensor(
    [0, 100, 300, 999],
    device=device
)

emb = time_embedding(test_t)

print("Forma de t:", test_t.shape)
print("Forma del embedding:", emb.shape)

Forma de t: torch.Size([4])
Forma del embedding: torch.Size([4, 64])


Bloques de la red

In [28]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.norm1 = nn.GroupNorm(
            8,
            out_channels
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.norm2 = nn.GroupNorm(
            8,
            out_channels
        )

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_channels)
        )

        if in_channels != out_channels:
            self.residual = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x, t_emb):

        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_information = self.time_mlp(t_emb)
        time_information = time_information[:, :, None, None]

        h = h + time_information

        h = self.conv2(h)
        h = self.norm2(h)

        h = h + self.residual(x)

        return F.silu(h)

U-net

In [29]:
class SimpleUNet(nn.Module):
    def __init__(self, time_dim=64):
        super().__init__()

        # Embedding temporal
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # Entrada:
        # En Fourier usamos dos canales:
        # canal 0 = parte real
        # canal 1 = parte imaginaria
        self.input_conv = nn.Conv2d(
            2,
            64,
            kernel_size=3,
            padding=1
        )

        # Encoder
        self.block1 = ResBlock(
            64,
            64,
            time_dim
        )

        self.down1 = nn.Conv2d(
            64,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.block2 = ResBlock(
            128,
            128,
            time_dim
        )

        self.down2 = nn.Conv2d(
            128,
            256,
            kernel_size=4,
            stride=2,
            padding=1
        )

        # Parte más profunda
        self.middle = ResBlock(
            256,
            256,
            time_dim
        )

        # Decoder
        self.up1 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.block3 = ResBlock(
            256,
            128,
            time_dim
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.block4 = ResBlock(
            128,
            64,
            time_dim
        )

        # Salida:
        # La U-Net predice el ruido en Fourier, por lo que necesitamos
        # dos canales: parte real e imaginaria.
        self.output_conv = nn.Conv2d(
            64,
            2,
            kernel_size=1
        )

    def forward(self, x, t):

        t_emb = self.time_embedding(t)

        # Entrada
        x = self.input_conv(x)

        # Encoder
        skip1 = self.block1(x, t_emb)

        x = self.down1(skip1)

        skip2 = self.block2(x, t_emb)

        x = self.down2(skip2)

        # Centro
        x = self.middle(x, t_emb)

        # Decoder
        x = self.up1(x)

        x = torch.cat(
            [x, skip2],
            dim=1
        )

        x = self.block3(x, t_emb)

        x = self.up2(x)

        x = torch.cat(
            [x, skip1],
            dim=1
        )

        x = self.block4(x, t_emb)

        return self.output_conv(x)

Red: Prueba de formas

In [30]:
model = SimpleUNet().to(device)

x0, _ = next(iter(loader))
x0 = x0.to(device)

X0 = image_to_fourier(x0)

t = torch.randint(
    0,
    T,
    (x0.shape[0],),
    device=device
)

Xt, fourier_noise = forward_diffusion_fourier(
    X0,
    t
)

# Predicción del ruido en Fourier
predicted_noise = model(
    Xt,
    t
)

print("x0:", x0.shape)
print("X0:", X0.shape)
print("Xt:", Xt.shape)
print("Ruido Fourier real:", fourier_noise.shape)
print("Ruido Fourier predicho:", predicted_noise.shape)

x0: torch.Size([64, 1, 28, 28])
X0: torch.Size([64, 2, 28, 28])
Xt: torch.Size([64, 2, 28, 28])
Ruido Fourier real: torch.Size([64, 2, 28, 28])
Ruido Fourier predicho: torch.Size([64, 2, 28, 28])


Función de pérdida

In [31]:
loss = F.mse_loss(
    predicted_noise,
    fourier_noise
)

print("Loss:", loss.item())

if predicted_noise.shape == fourier_noise.shape:
    print(
        "Las formas coinciden. "
        "La pérdida en Fourier se puede calcular correctamente."
    )

Loss: 0.7499721050262451
Las formas coinciden. La pérdida en Fourier se puede calcular correctamente.


Prueba de entrenamiento

In [32]:
# Reiniciamos el modelo para que esta prueba comience desde cero.
model = SimpleUNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

model.train()

# Tomamos un solo batch de 64 imágenes
x0, _ = next(iter(loader))
x0 = x0.to(device)

# Las imágenes originales se transforman a Fourier una vez.
X0 = image_to_fourier(x0)

loss_history = []

for step in range(300):

    # 1. Elegir un tiempo aleatorio para cada imagen
    t = torch.randint(
        0,
        T,
        (x0.shape[0],),
        device=device
    )

    # 2. Aplicar el proceso forward en Fourier
    Xt, fourier_noise = forward_diffusion_fourier(
        X0,
        t
    )

    # 3. La U-Net intenta predecir el ruido en Fourier
    predicted_noise = model(
        Xt,
        t
    )

    # 4. Comparar ruido Fourier real y ruido Fourier predicho
    loss = F.mse_loss(
        predicted_noise,
        fourier_noise
    )

    # 5. Borrar gradientes de la iteración anterior
    optimizer.zero_grad()

    # 6. Calcular nuevos gradientes
    loss.backward()

    # 7. Actualizar los parámetros de la red
    optimizer.step()

    # Guardar pérdida
    loss_history.append(loss.item())

    if step % 25 == 0:
        print(
            f"Step {step:3d} | "
            f"Loss Fourier: {loss.item():.4f}"
        )


plt.figure(figsize=(8, 5))

plt.plot(loss_history)

plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Entrenamiento de prueba en Fourier sobre un batch")

plt.grid()

plt.savefig(
    "fourier_batch_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Prueba de entrenamiento sobre un batch guardada.")


Step   0 | Loss Fourier: 0.8083
Step  25 | Loss Fourier: 0.1598
Step  50 | Loss Fourier: 0.1302
Step  75 | Loss Fourier: 0.0686
Step 100 | Loss Fourier: 0.0717
Step 125 | Loss Fourier: 0.0728
Step 150 | Loss Fourier: 0.0598
Step 175 | Loss Fourier: 0.0582
Step 200 | Loss Fourier: 0.0442
Step 225 | Loss Fourier: 0.0443
Step 250 | Loss Fourier: 0.0379
Step 275 | Loss Fourier: 0.0363
Prueba de entrenamiento sobre un batch guardada.


Entrenamiento con MNIST

In [33]:
# Reiniciamos otra vez el modelo.
# La prueba de 300 steps anterior NO afecta al entrenamiento completo.
model = SimpleUNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

epoch_losses = []

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for x0, _ in loader:

        # Mover imágenes al dispositivo
        x0 = x0.to(device)

        # Imagen -> Fourier
        X0 = image_to_fourier(x0)

        # Elegir un tiempo aleatorio para cada imagen
        t = torch.randint(
            0,
            T,
            (x0.shape[0],),
            device=device
        )

        # Proceso forward en Fourier
        Xt, fourier_noise = forward_diffusion_fourier(
            X0,
            t
        )

        # La U-Net predice el ruido en Fourier
        predicted_noise = model(
            Xt,
            t
        )

        # MSE entre ruido Fourier verdadero y predicho
        loss = F.mse_loss(
            predicted_noise,
            fourier_noise
        )

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    # Loss promedio del epoch
    average_loss = total_loss / len(loader)

    epoch_losses.append(average_loss)

    print(
        f"Épocas {epoch + 1:2d}/{EPOCHS} | "
        f"Pérdida Fourier promedio: {average_loss:.6f}"
    )


Épocas  1/30 | Pérdida Fourier promedio: 0.054159
Épocas  2/30 | Pérdida Fourier promedio: 0.039608
Épocas  3/30 | Pérdida Fourier promedio: 0.036063
Épocas  4/30 | Pérdida Fourier promedio: 0.034318
Épocas  5/30 | Pérdida Fourier promedio: 0.032252
Épocas  6/30 | Pérdida Fourier promedio: 0.030758
Épocas  7/30 | Pérdida Fourier promedio: 0.029586
Épocas  8/30 | Pérdida Fourier promedio: 0.028606
Épocas  9/30 | Pérdida Fourier promedio: 0.027927
Épocas 10/30 | Pérdida Fourier promedio: 0.027724
Épocas 11/30 | Pérdida Fourier promedio: 0.026692
Épocas 12/30 | Pérdida Fourier promedio: 0.026774
Épocas 13/30 | Pérdida Fourier promedio: 0.026002
Épocas 14/30 | Pérdida Fourier promedio: 0.025644
Épocas 15/30 | Pérdida Fourier promedio: 0.025445
Épocas 16/30 | Pérdida Fourier promedio: 0.025236
Épocas 17/30 | Pérdida Fourier promedio: 0.025293
Épocas 18/30 | Pérdida Fourier promedio: 0.024704
Épocas 19/30 | Pérdida Fourier promedio: 0.024712
Épocas 20/30 | Pérdida Fourier promedio: 0.024375


Curva de entrenamiento

In [34]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    epoch_losses,
    marker="o"
)

plt.xlabel("Épocas")
plt.ylabel("MSE pérdida promedio")
plt.title("Entrenamiento del modelo sobre MNIST en Fourier")

plt.grid()

plt.savefig(
    "mnist_fourier_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Curva de entrenamiento Fourier guardada.")

Curva de entrenamiento Fourier guardada.


Guardar modelo

In [35]:
torch.save(
    model.state_dict(),
    "mnist_fourier_diffusion_model.pt"
)

print("Modelo Fourier guardado.")

Modelo Fourier guardado.


Cargar modelo

In [36]:
model = SimpleUNet().to(device)

model.load_state_dict(
    torch.load(
        "mnist_fourier_diffusion_model.pt",
        map_location=device
    )
)

model.eval()

print("Modelo Fourier cargado correctamente.")

Modelo Fourier cargado correctamente.


Proceso backward en fourier

In [37]:
@torch.no_grad()
def sample_reverse_fourier(model, n_samples=16):

    model.eval()

    # Empezamos desde ruido gaussiano puro en el espacio real.
    spatial_noise = torch.randn(
        n_samples,
        1,
        28,
        28,
        device=device
    )

    # Lo transformamos a Fourier.
    X = image_to_fourier(spatial_noise)

    # Guardaremos algunos estados intermedios en Fourier.
    snapshots = {}

    save_times = [999, 750, 500, 250, 0]

    for timestep in reversed(range(T)):

        # Todos los elementos están en el mismo timestep.
        t = torch.full(
            (n_samples,),
            timestep,
            device=device,
            dtype=torch.long
        )

        # La U-Net estima el ruido presente en Fourier.
        predicted_noise = model(
            X,
            t
        )

        # La red no tiene incorporada explícitamente la restricción de
        # simetría hermítica. Podemos proyectar su salida al subespacio
        # de espectros compatibles con imágenes reales.
        if ENFORCE_REAL_SPECTRUM:
            predicted_noise = project_to_real_fourier(
                predicted_noise
            )

        beta_t = betas[timestep]
        alpha_t = alphas[timestep]
        alpha_bar_t = alpha_bars[timestep]

        # Media de p(X_{t-1} | X_t)
        mean = (
            1 / torch.sqrt(alpha_t)
        ) * (
            X
            -
            beta_t
            / torch.sqrt(1 - alpha_bar_t)
            * predicted_noise
        )

        if timestep > 0:

            alpha_bar_previous = alpha_bars[
                timestep - 1
            ]

            posterior_variance = (
                beta_t
                * (1 - alpha_bar_previous)
                / (1 - alpha_bar_t)
            )

            # El ruido estocástico también se genera primero como una
            # imagen real y luego se lleva al espacio de Fourier.
            z_spatial = torch.randn(
                n_samples,
                1,
                28,
                28,
                device=device
            )

            z = image_to_fourier(
                z_spatial
            )

            X = (
                mean
                +
                torch.sqrt(posterior_variance) * z
            )

        else:
            X = mean

        # Opcionalmente preservamos la consistencia con imágenes reales
        # después de cada paso.
        if ENFORCE_REAL_SPECTRUM:
            X = project_to_real_fourier(X)

        if timestep in save_times:
            snapshots[timestep] = X.detach().cpu()

    return X, snapshots


Generar muestras en fourier

In [38]:
# Semilla fija para poder comparar distintas corridas/modelos.
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

generated_fourier, snapshots = sample_reverse_fourier(
    model,
    n_samples=16
)

# Fourier -> imagen
generated_images = fourier_to_image(
    generated_fourier
)

# Medimos cuánto residuo imaginario quedaría si no tomáramos solamente
# la parte real de la IFFT.
generated_complex = fourier_to_complex_image(
    generated_fourier
)

final_imaginary_residual = torch.mean(
    torch.abs(generated_complex.imag)
)

print(
    "Residuo imaginario promedio final:",
    final_imaginary_residual.item()
)

Residuo imaginario promedio final: 5.095987276604319e-08


Visualización de imágenes

In [39]:
plt.figure(figsize=(8, 8))

for i in range(16):

    plt.subplot(4, 4, i + 1)

    image = generated_images[
        i
    ].detach().cpu()

    image = (image + 1) / 2
    image = image.clamp(0, 1)

    plt.imshow(
        image.squeeze(),
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()

plt.savefig(
    "generated_mnist_fourier.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Imágenes generadas desde Fourier guardadas.")

Imágenes generadas desde Fourier guardadas.


Visualización del proceso backward

In [40]:
times_to_show = [999, 750, 500, 250, 0]

plt.figure(figsize=(12, 3))

for i, timestep in enumerate(times_to_show):

    X_snapshot = snapshots[
        timestep
    ].to(device)

    image = fourier_to_image(
        X_snapshot
    )[0]

    image = image.detach().cpu()

    image = (image + 1) / 2
    image = image.clamp(0, 1)

    plt.subplot(
        1,
        5,
        i + 1
    )

    plt.imshow(
        image.squeeze(),
        cmap="gray"
    )

    plt.title(
        f"t={timestep}"
    )

    plt.axis("off")

plt.tight_layout()

plt.savefig(
    "reverse_process_fourier.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Proceso reverse Fourier guardado.")

Proceso reverse Fourier guardado.


Visualización del espectro final generado

In [41]:
generated_magnitudes = fourier_magnitude(
    generated_fourier
).detach().cpu()

plt.figure(figsize=(8, 8))

for i in range(16):

    plt.subplot(
        4,
        4,
        i + 1
    )

    plt.imshow(
        generated_magnitudes[i, 0],
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()

plt.savefig(
    "generated_fourier_magnitudes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Magnitudes de Fourier generadas guardadas.")


print("\nProceso completo terminado:")
print("Imagen -> Fourier -> DDPM -> Reverse en Fourier -> IFFT -> Imagen")

Magnitudes de Fourier generadas guardadas.

Proceso completo terminado:
Imagen -> Fourier -> DDPM -> Reverse en Fourier -> IFFT -> Imagen
